# Notebook 01 - Human Preference Annotation UI

**RLHF Preference Trainer** - Step 1 of 5

Annotate which GPT-2 response is better across 4 dimensions: **Helpfulness, Factuality, Safety, Fluency**

Saves to `data/preferences.csv`. Target ~1,200 pairs.

> Pairs are generated **one at a time** after each annotation - UI launches in seconds.

---

In [ ]:
!pip install -q transformers gradio pandas torch accelerate sentencepiece
print('Dependencies installed')

In [ ]:
import os, sys

if 'google.colab' in str(get_ipython()):
    if not os.path.exists('rlhf-preference-trainer'):
        !git clone https://github.com/sharma614/rlhf-preference-trainer.git
    os.chdir('rlhf-preference-trainer')
else:
    os.chdir(os.path.abspath(os.path.join(os.getcwd(), '..')))

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print(f'Working dir: {os.getcwd()}')

In [ ]:
import datetime, random
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.data_utils import SEED_PROMPTS, init_preferences_csv, save_annotation, generate_response
from src.ppo_config import MODEL_NAME, PREFERENCES_CSV

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device} | Model: {MODEL_NAME}')

In [ ]:
print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
).to(device)
model.eval()
print(f'{MODEL_NAME} loaded')

In [ ]:
# Only 3 pairs generated upfront (~30s). New pairs generate one at a time after each annotation.
os.makedirs('data', exist_ok=True)
init_preferences_csv(PREFERENCES_CSV)

def _make_pair(seed):
    prompt = random.choice(SEED_PROMPTS)
    return {
        'prompt': prompt,
        'response_a': generate_response(prompt, model, tokenizer, max_new_tokens=120, seed=seed),
        'response_b': generate_response(prompt, model, tokenizer, max_new_tokens=120, seed=seed + 500),
    }

print('Generating 3 seed pairs...')
ALL_PAIRS = [_make_pair(i) for i in range(3)]
print(f'Ready. More pairs will generate automatically as you annotate.')

In [ ]:
STATE = {
    'pair_idx': 0,
    'pairs': ALL_PAIRS,
    'session_count': 0,
    'annotator_id': f'annotator_{random.randint(1000, 9999)}',
    'gen_seed': 100,
}

def get_current_pair():
    while STATE['pair_idx'] >= len(STATE['pairs']):
        STATE['pairs'].append(_make_pair(STATE['gen_seed']))
        STATE['gen_seed'] += 1
    return STATE['pairs'][STATE['pair_idx']]

def get_existing_count():
    try:
        return len(pd.read_csv(PREFERENCES_CSV))
    except Exception:
        return 0

print(f'Annotator ID: {STATE["annotator_id"]}')
print(f'Existing annotations: {get_existing_count()}')

In [ ]:
import gradio as gr

CUSTOM_CSS = """
.response-box {
    border: 2px solid #e0e0e0;
    border-radius: 12px;
    padding: 16px;
    min-height: 200px;
    background: #fafafa;
    font-size: 14px;
    line-height: 1.6;
}
.label-a { border-color: #4A90D9 !important; }
.label-b { border-color: #E67E22 !important; }
"""

def load_pair():
    pair = get_current_pair()
    count = get_existing_count()
    progress = min(100, int(count / 1200 * 100))
    return (
        f"**Prompt:**\n\n> {pair['prompt']}",
        pair['response_a'],
        pair['response_b'],
        f"{count} / 1200 annotated ({progress}%)",
        gr.update(value=3),
        gr.update(value=3),
        gr.update(value=3),
        gr.update(value=3),
    )

def submit_annotation(preferred, helpfulness, factuality, safety, fluency):
    if preferred is None:
        return load_pair() + ('Please select a preference first.',)
    pair = get_current_pair()
    save_annotation({
        'prompt': pair['prompt'],
        'response_a': pair['response_a'],
        'response_b': pair['response_b'],
        'preferred': preferred,
        'helpfulness_score': helpfulness,
        'factuality_score': factuality,
        'safety_score': safety,
        'fluency_score': fluency,
        'annotator_id': STATE['annotator_id'],
        'timestamp': datetime.datetime.now().isoformat(),
    }, PREFERENCES_CSV)
    STATE['pair_idx'] += 1
    STATE['session_count'] += 1
    return load_pair() + (f"Saved! Session total: {STATE['session_count']}",)

def skip_pair():
    STATE['pair_idx'] += 1
    return load_pair() + ('Skipped.',)

with gr.Blocks(css=CUSTOM_CSS, title='RLHF Annotation UI', theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # RLHF Preference Annotation Interface
    Read the prompt, rate both responses, pick your preference, then click **Submit**.
    """)

    counter_display = gr.Markdown(value='Loading...')
    prompt_display  = gr.Markdown(value='Loading...')

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown('### Response A')
            resp_a = gr.Textbox(label='Response A', lines=10, interactive=False,
                                elem_classes=['response-box', 'label-a'])
        with gr.Column(scale=1):
            gr.Markdown('### Response B')
            resp_b = gr.Textbox(label='Response B', lines=10, interactive=False,
                                elem_classes=['response-box', 'label-b'])

    gr.Markdown('---')
    gr.Markdown('### Rate the **preferred** response (1=Poor, 5=Excellent)')

    with gr.Row():
        helpfulness_sl = gr.Slider(1, 5, value=3, step=1, label='Helpfulness')
        factuality_sl  = gr.Slider(1, 5, value=3, step=1, label='Factuality')
    with gr.Row():
        safety_sl  = gr.Slider(1, 5, value=3, step=1, label='Safety')
        fluency_sl = gr.Slider(1, 5, value=3, step=1, label='Fluency')

    gr.Markdown('### Overall Preference')
    preference = gr.Radio(choices=['A', 'B', 'Tie'], label='Which response do you prefer?', value=None)
    status_msg = gr.Markdown(value='')

    with gr.Row():
        submit_btn = gr.Button('Submit & Next', variant='primary', scale=3)
        skip_btn   = gr.Button('Skip', variant='secondary', scale=1)

    slider_outputs = [helpfulness_sl, factuality_sl, safety_sl, fluency_sl]
    all_outputs    = [prompt_display, resp_a, resp_b, counter_display] + slider_outputs

    submit_btn.click(fn=submit_annotation,
                     inputs=[preference, helpfulness_sl, factuality_sl, safety_sl, fluency_sl],
                     outputs=all_outputs + [status_msg])
    skip_btn.click(fn=skip_pair, inputs=[], outputs=all_outputs + [status_msg])
    demo.load(fn=load_pair, outputs=all_outputs)

print('Gradio app built')

In [ ]:
demo.launch(share=True, debug=False)

In [ ]:
import pandas as pd
from src.ppo_config import PREFERENCES_CSV

try:
    df = pd.read_csv(PREFERENCES_CSV)
    print(f'Total annotations : {len(df)}')
    print(f'Unique annotators : {df["annotator_id"].nunique()}')
    print('\nPreference distribution:')
    for pref, cnt in df['preferred'].value_counts().items():
        print(f'  {pref}: {cnt} ({100*cnt/len(df):.1f}%)')
    print('\nMean Scores:')
    for dim in ['helpfulness_score', 'factuality_score', 'safety_score', 'fluency_score']:
        print(f'  {dim.replace("_score","").title():15s}: {df[dim].mean():.2f}/5.00')
    display(df[['prompt', 'preferred', 'helpfulness_score', 'fluency_score']].head(5))
except FileNotFoundError:
    print('No annotations yet.')

In [ ]:
import os
from src.ppo_config import PREFERENCES_CSV

assert os.path.exists(PREFERENCES_CSV), f'{PREFERENCES_CSV} not found!'
df_check = pd.read_csv(PREFERENCES_CSV)
for col in ['prompt', 'response_a', 'response_b', 'preferred',
            'helpfulness_score', 'factuality_score', 'safety_score', 'fluency_score']:
    assert col in df_check.columns, f'Missing column: {col}'
print(f'CSV has {len(df_check)} rows, all columns present.')
print('Next: Run 02_reward_model_training.ipynb')